In [1]:
## fetching the dataset 
import pandas as pd 

df = pd.read_csv('data/fe_bank.csv')
## display the head of the dataset

df.head()

,age,job,marital,education,default,housing,loan,contact,day,month,pdays,poutcome,deposit,balance_log,campaign_log,previous_log
0,59,admin.,married,secondary,no,yes,no,unknown,5,may,-1,unknown,yes,7.759614,0.693147,0.0
1,56,admin.,married,secondary,no,no,no,unknown,5,may,-1,unknown,yes,3.828641,0.693147,0.0
2,41,technician,married,secondary,no,yes,no,unknown,5,may,-1,unknown,yes,7.147559,0.693147,0.0
3,55,services,married,secondary,no,yes,no,unknown,5,may,-1,unknown,yes,7.814803,0.693147,0.0
4,54,admin.,married,tertiary,no,no,no,unknown,5,may,-1,unknown,yes,5.220356,1.098612,0.0


In [2]:
X = df.drop('deposit',axis=1)
y = df['deposit']

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
## Segregating numerical and categorical columns
num_columns = X.select_dtypes(include='number').columns
cat_columns = X.select_dtypes(include='object').columns

print(f"Numerical Columns: \n{num_columns}")
print()
print(f"Categorical Columns: \n{cat_columns}")

Numerical Columns: 
Index(['age', 'day', 'pdays', 'balance_log', 'campaign_log', 'previous_log'],
      dtype='str')

Categorical Columns: 
Index(['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
       'month', 'poutcome'],
      dtype='str')


## creating numerical and categorical pipeline

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder

num_pipeline = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='median')),
        ('scaler',StandardScaler())
    ]
)

cat_pipeline = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
    ]
)

preprocessor = ColumnTransformer([
    ('num_pipeline',num_pipeline,num_columns),
    ('cat_pipeline',cat_pipeline,cat_columns)
])

In [6]:
## Using train test split to split the dataset 
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=42,stratify=df['deposit'])

In [7]:
X_train = pd.DataFrame(preprocessor.fit_transform(X_train),columns=preprocessor.get_feature_names_out())
X_test = pd.DataFrame(preprocessor.transform(X_test),columns=preprocessor.get_feature_names_out())

In [8]:
## Using label encoder to transform target feature
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [9]:
## evaluating the model
from sklearn.metrics import accuracy_score,confusion_matrix,recall_score,precision_score,f1_score,roc_auc_score
import numpy as np 

def eval_model(true,pred,proba):
    accuracy = accuracy_score(true,pred)
    confi_matrix = confusion_matrix(true,pred)
    precision = precision_score(true,pred)
    recall = recall_score(true,pred)
    f1 = f1_score(true,pred)

    if proba is not None:
        roc_auc = roc_auc_score(true,proba)
    else:
        roc_auc = np.nan

    return accuracy,confi_matrix,precision,recall,f1,roc_auc

In [10]:
## Training the model and finding the best one (RandomSearch CV)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, uniform, randint

In [11]:
models = {
    "LogisticRegression": LogisticRegression(),
    "DecisionTreeClassifier": DecisionTreeClassifier(),
    "RandomForestClassifier": RandomForestClassifier(),
    "AdaBoostClassifier": AdaBoostClassifier(),
    "GradientBoostingClassifier": GradientBoostingClassifier(),
    "SVC":SVC(),
    "XGBClassifier":XGBClassifier(),
    "CatBoostClassifier":CatBoostClassifier()
}

param_distributions = {

    "LogisticRegression": {
        "C": loguniform(1e-3, 1e2),
        "penalty": ["l1", "l2"],
        "solver": ["liblinear", "saga"],
        "class_weight": [None, "balanced"],
        "max_iter": [1000, 2000, 3000]
    },

    "DecisionTreeClassifier": {
        "criterion": ["gini", "entropy", "log_loss"],
        "max_depth": [None, 5, 10, 15, 20, 30],
        "min_samples_split": randint(2, 30),
        "min_samples_leaf": randint(1, 20),
        "max_features": [None, "sqrt", "log2"],
        "class_weight": [None, "balanced"]
    },

    "RandomForestClassifier": {
        "n_estimators": randint(100, 800),
        "criterion": ["gini", "entropy", "log_loss"],
        "max_depth": [None, 10, 20, 30, 40],
        "min_samples_split": randint(2, 30),
        "min_samples_leaf": randint(1, 15),
        "max_features": ["sqrt", "log2", None],
        "bootstrap": [True, False],
        "class_weight": [None, "balanced", "balanced_subsample"]
    },

    "GradientBoostingClassifier": {
        "n_estimators": randint(50, 500),
        "learning_rate": loguniform(0.01, 0.3),
        "max_depth": randint(1, 6),
        "min_samples_split": randint(2, 30),
        "min_samples_leaf": randint(1, 20),
        "subsample": uniform(0.6, 0.4),
        "max_features": [None, "sqrt", "log2"]
    },

    "AdaBoostClassifier": {
        "n_estimators": randint(50, 500),
        "learning_rate": loguniform(0.01, 2.0)
    },

    "SVC": {
        "C": loguniform(1e-3, 1e2),
        "kernel": ["linear", "rbf", "poly"],
        "gamma": ["scale", "auto"],
        "class_weight": [None, "balanced"],
        "degree": randint(2, 5)
    },

    "XGBClassifier": {
        "n_estimators": randint(100, 800),
        "max_depth": randint(3, 11),
        "learning_rate": loguniform(0.01, 0.3),
        "min_child_weight": randint(1, 15),
        "gamma": uniform(0, 5),
        "subsample": uniform(0.6, 0.4),
        "colsample_bytree": uniform(0.6, 0.4),
        "reg_alpha": loguniform(1e-4, 10),
        "reg_lambda": loguniform(1e-3, 20),
        "scale_pos_weight": [1, 2, 3, 4, 5]
    },

    "CatBoostClassifier": {
        "iterations": randint(200, 1000),
        "depth": randint(4, 11),
        "learning_rate": loguniform(0.01, 0.3),
        "l2_leaf_reg": loguniform(1, 10),
        "random_strength": uniform(0, 2),
        "bagging_temperature": uniform(0, 2),
        "border_count": [32, 64, 128, 254],
        "auto_class_weights": [None, "Balanced", "SqrtBalanced"]
    }
}

report = {}
best_score = -1
best_model = None
best_params = None

for model_name, model in models.items():
    print(f"{model_name} evaluation started..")
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions[model_name],
        verbose=1,
        n_jobs=-1,
        n_iter=50,
        scoring='roc_auc',
        random_state=42
    )

    search.fit(X_train,y_train)

    cv_score = search.best_score_
    tuner = search.best_estimator_
    tuner_params = search.best_params_

    print(f"{model_name} best cv score: {cv_score} and best params: {tuner_params}")

    if cv_score > best_score:
        best_score = cv_score
        best_model = tuner
        best_params = tuner_params

    y_pred = search.predict(X_test)

    if hasattr(model,'predict_proba'):
        y_proba = search.predict_proba(X_test)[:,1]
    elif hasattr(model,'decision_function'):
        y_proba = search.decision_function(X_test)
    else:
        y_proba = None

    accuracy,confi_matrix,precision,recall,f1,roc_auc = eval_model(y_test,y_pred,y_proba)

    report[model_name] = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1":f1,
        "roc_auc":roc_auc
    }
    print()
    print(f"{model_name} confusion matrix: \n{str(confi_matrix)}")
    print('-'*10)

print(f"Best model found: {best_model} with best score: {best_score} and best params: {best_params}")
report_df = pd.DataFrame(report).T.sort_values(by='roc_auc',ascending=False).reset_index().rename(columns={'index':'model'})
report_df.head(10)

LogisticRegression evaluation started..
Fitting 5 folds for each of 50 candidates, totalling 250 fits
LogisticRegression best cv score: 0.7654602208758874 and best params: {'C': np.float64(0.21568228019263963), 'class_weight': None, 'max_iter': 3000, 'penalty': 'l1', 'solver': 'liblinear'}

LogisticRegression confusion matrix: 
[[1203  266]
 [ 546  776]]
----------
DecisionTreeClassifier evaluation started..
Fitting 5 folds for each of 50 candidates, totalling 250 fits
DecisionTreeClassifier best cv score: 0.7457475529469739 and best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': 10, 'max_features': None, 'min_samples_leaf': 7, 'min_samples_split': 27}

DecisionTreeClassifier confusion matrix: 
[[1192  277]
 [ 579  743]]
----------
RandomForestClassifier evaluation started..
Fitting 5 folds for each of 50 candidates, totalling 250 fits
RandomForestClassifier best cv score: 0.7898923578645615 and best params: {'bootstrap': False, 'class_weight': None, 'criterion': 'log

,model,accuracy,precision,recall,f1,roc_auc
0,CatBoostClassifier,0.743461,0.778493,0.640696,0.702905,0.792274
1,XGBClassifier,0.742745,0.784369,0.630106,0.698826,0.791247
2,GradientBoostingClassifier,0.740236,0.780282,0.628593,0.696271,0.790539
3,RandomForestClassifier,0.731637,0.769014,0.619516,0.686217,0.787647
4,SVC,0.738445,0.790196,0.609682,0.688301,0.781651
5,AdaBoostClassifier,0.722322,0.755847,0.611195,0.675868,0.772312
6,LogisticRegression,0.709065,0.744722,0.586989,0.656514,0.758947
7,DecisionTreeClassifier,0.693300,0.728431,0.562027,0.634500,0.740403
